# RTM3D/KM3D ResNet-18 — decoded ONNX parity diagnostic

**Revision:** `RTM3D-KM3D-ONNX-DECODED-PARITY-2026-10-08-r1`

This follows the raw-head ONNX check. It runs the same 16 frozen KITTI images through the PyTorch reference heads and ONNX Runtime CPU, then sends both output sets through the same pinned RTM3D calibration-based decoder. It measures whether the small raw `hps` differences alter decoded keypoints, boxes, 3D dimensions, yaw, or location.

This is a **diagnostic**, not an accuracy gate: it does not train, score KITTI AP, or qualify deployment. RTM3D's upstream geometry decoder uses CUDA, so select a Colab GPU runtime before running. No phone connection is needed yet.

## 1. Mount Drive, update project code, and check the runtime

Run this first. It installs only ONNX Runtime/OpenCV if missing; it does not replace Colab's PyTorch/CUDA stack.

In [ ]:
from pathlib import Path
from datetime import datetime
import importlib.util, json, subprocess, sys, zipfile
from google.colab import drive, files
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/mobile_adas3d')
PROJECT_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', PROJECT_URL, str(PROJECT_DIR)], check=True)
origin = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
if origin.startswith('git@github.com:'): origin = 'https://github.com/' + origin.split(':', 1)[1]
if origin.rstrip('/').removesuffix('.git').lower() != 'https://github.com/ali-rt/mobile_adas3d': raise RuntimeError(f'Unexpected project origin: {origin!r}')
if not (PROJECT_DIR / '.git').exists(): raise RuntimeError(f'{PROJECT_DIR} is not a Git checkout; choose a fresh path.')
branch = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'branch', '--show-current'], text=True).strip()
if branch != 'main': raise RuntimeError(f'Expected main, found {branch!r}; preserve this checkout and use a clean main checkout.')
dirty = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'status', '--porcelain', '--untracked-files=no'], text=True).strip()
if dirty: raise RuntimeError('MobileADAS3D has tracked edits; preserve them and use a clean checkout.')
subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
SCRIPT = PROJECT_DIR / 'scripts/audit_rtm3d_km3d_res18_onnx_decoded.py'
if not SCRIPT.is_file(): raise FileNotFoundError(f'{SCRIPT} is missing from main; confirm the decoded-parity revision is published.')

missing = []
if importlib.util.find_spec('onnxruntime') is None: missing.append('onnxruntime')
if importlib.util.find_spec('cv2') is None: missing.append('opencv-python-headless')
if missing: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
import cv2, onnxruntime as ort, torch
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime > Change runtime type > T4 GPU (or another GPU), then rerun from this cell.')
print('MobileADAS3D main:', subprocess.check_output(['git','-C',str(PROJECT_DIR),'rev-parse','--short','HEAD'],text=True).strip())
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0))
print('ONNX Runtime:', ort.__version__, '| providers:', ort.get_available_providers(), '| OpenCV:', cv2.__version__)

## 2. Select the exact ONNX bundle, pinned RTM3D source, and matching KITTI images

The bundle is normally already in the Drive folder created by the ONNX export notebook. If it is not found, this cell asks you to upload `rtm3d_km3d_res18_onnx_export.zip`. Existing RTM3D folders are preserved; a clean pinned checkout gets a fresh suffix if needed.

In [ ]:
from scripts.run_rtm3d_km3d_res18_smoke import RTM3D_COMMIT, RTM3D_URL, find_sample_paths
EXPORT_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/rtm3d_km3d_res18_r1')
EXPECTED_COMMIT = '888c379e79d8a6d134f06a9b7d669118679e06dc'
EXPECTED_CHECKPOINT_SHA256 = '5fa355845f79c1afeffab427de32933758e5b4c1e7c9ec19a94a13737691d05b'
if RTM3D_COMMIT != EXPECTED_COMMIT: raise RuntimeError(f'Unexpected pinned RTM3D commit constant: {RTM3D_COMMIT}')
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
bundles = sorted(EXPORT_ROOT.glob('onnx_export_r1_*/rtm3d_km3d_res18_onnx_export.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
if not bundles:
    print('No export bundle found in Drive. Upload the ZIP from the ONNX export step.')
    uploaded = files.upload()
    if len(uploaded) != 1: raise RuntimeError(f'Expected one ONNX export ZIP, received {len(uploaded)} files')
    uploaded_name = next(iter(uploaded))
    BUNDLE = EXPORT_ROOT / uploaded_name
    BUNDLE.write_bytes(uploaded[uploaded_name])
else:
    BUNDLE = bundles[0]
with zipfile.ZipFile(BUNDLE) as archive:
    export_report = json.loads(archive.read('rtm3d_onnx_export.json'))
    raw_report = json.loads(archive.read('onnxruntime_cpu_parity.json'))
if export_report.get('source_commit') != EXPECTED_COMMIT: raise RuntimeError(f'Bundle source commit mismatch: {export_report.get("source_commit")}')
if export_report.get('checkpoint_sha256') != EXPECTED_CHECKPOINT_SHA256: raise RuntimeError('Bundle checkpoint differs from the reviewed ResNet-18 checkpoint')
if not export_report.get('export_complete') or not export_report.get('onnx_checker_passed'): raise RuntimeError('Bundle does not contain a complete, ONNX-checker-validated export')
if raw_report.get('onnx_sha256') != export_report.get('onnx_sha256'): raise RuntimeError('Export and raw-parity reports refer to different ONNX files')
if raw_report.get('parity_fixture_sha256') != export_report.get('parity_fixture_sha256'): raise RuntimeError('Export and raw-parity reports refer to different fixtures')
SAMPLE_IDS = export_report['input']['fixture_sample_ids']
if len(SAMPLE_IDS) != 16 or len(set(SAMPLE_IDS)) != 16: raise RuntimeError(f'Expected 16 unique parity fixtures, got {len(SAMPLE_IDS)}')
DATA_CANDIDATES = [Path('/content/kitti_m67'),Path('/content/kitti_m66'),Path('/content/kitti_m65'),Path('/content/kitti_m64'),Path('/content/kitti_m62'),Path('/content/kitti_m61'),Path('/content/kitti'),Path('/content/monodetr_kitti_a2'),Path('/content/drive/MyDrive/datasets/kitti')]
DATASET_ROOT = None
for candidate in DATA_CANDIDATES:
    try:
        for sample_id in SAMPLE_IDS: find_sample_paths(candidate, sample_id)
        DATASET_ROOT = candidate; break
    except FileNotFoundError:
        continue
if DATASET_ROOT is None: raise FileNotFoundError('Could not find all 16 fixture images and calibration files; add your KITTI root to DATA_CANDIDATES.')

def inspect_rtm3d(path):
    if not path.exists(): return False, 'missing'
    try:
        head = subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
        origin = subprocess.check_output(['git','-C',str(path),'remote','get-url','origin'],text=True,stderr=subprocess.DEVNULL).strip()
        dirty = subprocess.check_output(['git','-C',str(path),'status','--porcelain','--untracked-files=all'],text=True).strip()
    except subprocess.CalledProcessError: return False, 'not a Git checkout'
    problems = []
    if head != EXPECTED_COMMIT: problems.append(f'commit {head}')
    if origin.rstrip('/').removesuffix('.git') != RTM3D_URL.removesuffix('.git'): problems.append('unexpected origin')
    if dirty: problems.append('modified or untracked files')
    return not problems, '; '.join(problems) if problems else 'clean pinned checkout'

base = Path('/content/RTM3D_KM3D_R1_20261007_decoder')
RTM3D_REPO = base
while RTM3D_REPO.exists() and not inspect_rtm3d(RTM3D_REPO)[0]:
    RTM3D_REPO = base.with_name(base.name + '_fresh' + str(int(RTM3D_REPO.name.rsplit('_fresh',1)[-1]) + 1 if '_fresh' in RTM3D_REPO.name else 1))
if not RTM3D_REPO.exists():
    subprocess.run(['git','clone',RTM3D_URL,str(RTM3D_REPO)],check=True)
    subprocess.run(['git','-C',str(RTM3D_REPO),'checkout','--detach',EXPECTED_COMMIT],check=True)
ok, reason = inspect_rtm3d(RTM3D_REPO)
if not ok: raise RuntimeError(f'Pinned RTM3D source verification failed: {reason}')
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUTPUT = EXPORT_ROOT / f'decoded_parity_r1_{RUN_TAG}.json'
LOG_DIR = EXPORT_ROOT / 'colab_logs'; LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / f'decoded_parity_r1_{RUN_TAG}.log'
print('Bundle:', BUNDLE)
print('Raw-head CPU parity complete:', raw_report.get('complete'))
print('Raw-head failures:', [name for name, value in raw_report.get('per_head', {}).items() if not value.get('passed', False)])
print('RTM3D source:', RTM3D_REPO, '| commit:', EXPECTED_COMMIT)
print('KITTI root:', DATASET_ROOT, '| fixed fixtures:', SAMPLE_IDS)
print('Output:', OUTPUT)

## 3. Decode both runtimes with the same calibration-aware RTM3D decoder

This runs the 16 fixtures only. It does not lower the raw-head `2e-4` tolerance or turn that gate into a pass. Detections are paired by class and 2D-box IoU at 0.5; the report then summarizes decoded keypoint, box, and 3D geometry differences.

In [ ]:
command = [sys.executable, '-u', str(SCRIPT), '--bundle', str(BUNDLE), '--repo', str(RTM3D_REPO), '--dataset-root', str(DATASET_ROOT), '--output', str(OUTPUT)]
print('+', ' '.join(command), flush=True)
with LOG.open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True); log.write(line); log.flush()
    code = process.wait()
if code: raise RuntimeError(f'Decoded parity diagnostic exited {code}; full log: {LOG}')
report = json.loads(OUTPUT.read_text())
assert report['complete'] and report['interpretation_only_no_deployment_gate']
print('\nDecoded detections by class:')
for name, counts in report['counts_by_class'].items(): print(name, counts)
print('\nMatched-pair differences (reference vs ONNX CPU):')
for name, values in report['matched_pair_delta_distributions'].items(): print(f'{name}: {values}')
print('\nThis is not AP or an iPhone performance test.')
print('Report:', OUTPUT, '| Durable log:', LOG)
files.download(str(OUTPUT))

## What to send back

Send the downloaded `decoded_parity_r1_*.json`. If `hps` differs in the raw tensors but decoded keypoints, yaw, and location remain close, the raw tolerance may be stricter than the practical decoded effect for this 16-image diagnostic. If geometry changes materially or detections become unmatched, we keep the issue open and investigate before full validation. Neither outcome establishes KITTI accuracy, iPhone speed, or deployment readiness.